In [21]:
import cv2
import mediapipe as mp
import serial
import time

arduino = serial.Serial("COM9", 9600)
time.sleep(2)

mp_hands = mp.solutions.hands
mp_face = mp.solutions.face_mesh
mp_draw = mp.solutions.drawing_utils

hands = mp_hands.Hands(max_num_hands=2)
face = mp_face.FaceMesh(refine_landmarks=True)

cap = cv2.VideoCapture(0)


def count_fingers(hand_landmarks):
    tips = [4, 8, 12, 16, 20]
    fingers = 0

    if hand_landmarks:
        if hand_landmarks.landmark[tips[1]].y < hand_landmarks.landmark[tips[1] - 2].y:
            fingers += 1
        if hand_landmarks.landmark[tips[2]].y < hand_landmarks.landmark[tips[2] - 2].y:
            fingers += 1
        if hand_landmarks.landmark[tips[3]].y < hand_landmarks.landmark[tips[3] - 2].y:
            fingers += 1
        if hand_landmarks.landmark[tips[4]].y < hand_landmarks.landmark[tips[4] - 2].y:
            fingers += 1

    return fingers


def mouth_open(landmarks):
    if not landmarks:
        return False

    top = landmarks[13].y
    bottom = landmarks[14].y
    return abs(top - bottom) > 0.03


def get_gesture_command(right_fingers, left_fingers, right_closed, left_closed):
    if right_fingers == 1 and left_fingers == 0:
        return "O"   # Orange
    elif left_fingers == 1 and right_fingers == 0:
        return "B"   # Blue
    elif right_fingers >= 3 and left_fingers >= 3:
        return "G"   # Green
    elif right_closed and left_closed:
        return "W"   # White
    elif (right_fingers >= 3 and left_fingers == 0) or (
        left_fingers >= 3 and right_fingers == 0
    ):
        return "R"   # Red
    elif (right_closed and not left_closed) or (left_closed and not right_closed):
        return "Y"   # Yellow
    else:
        return "N"


def command_name(cmd):
    names = {
        "R": "RED",
        "Y": "YELLOW",
        "G": "GREEN",
        "W": "WHITE",
        "O": "ORANGE",
        "B": "BLUE",
        "N": "NO COLOR",
    }
    return names.get(cmd, "UNKNOWN")


last_cmd = "N"
mouth_was_open = False

while True:
    ret, frame = cap.read()
    if not ret:
        break

    frame = cv2.flip(frame, 1)
    rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)

    hand_results = hands.process(rgb)
    face_results = face.process(rgb)

    right_fingers = 0
    left_fingers = 0
    right_closed = False
    left_closed = False

    # رسم اليدين وتحليلها
    if hand_results.multi_hand_landmarks and hand_results.multi_handedness:
        for i, hand_landmarks in enumerate(hand_results.multi_hand_landmarks):
            mp_draw.draw_landmarks(frame, hand_landmarks, mp_hands.HAND_CONNECTIONS)

            label = hand_results.multi_handedness[i].classification[0].label
            fingers = count_fingers(hand_landmarks)

            x = int(hand_landmarks.landmark[0].x * frame.shape[1])
            y = int(hand_landmarks.landmark[0].y * frame.shape[0])

            cv2.putText(
                frame,
                label,
                (x - 30, y - 20),
                cv2.FONT_HERSHEY_SIMPLEX,
                0.7,
                (0, 255, 255),
                2,
            )

            if label == "Right":
                right_fingers = fingers
                right_closed = fingers == 0
            else:
                left_fingers = fingers
                left_closed = fingers == 0

    # اللون المتوقع الحالي من وضعية اليد
    preview_command = get_gesture_command(
        right_fingers, left_fingers, right_closed, left_closed
    )

    # فحص الفم
    mouth_is_open = False
    if face_results.multi_face_landmarks:
        face_landmarks = face_results.multi_face_landmarks[0].landmark
        mouth_is_open = mouth_open(face_landmarks)

    # الإدخال فقط عند فتح الفم (لحظة الفتح)
    command_to_send = "N"
    if mouth_is_open and not mouth_was_open:
        if preview_command != "N":
            command_to_send = preview_command

    mouth_was_open = mouth_is_open

    # أرسل فقط عند وجود أمر جديد
    if command_to_send != "N" and command_to_send != last_cmd:
        arduino.write(command_to_send.encode())
        last_cmd = command_to_send

    # لو أرسلنا نفس اللون مرة ثانية بعد فتح فم جديد، لازم ينرسل
    if not mouth_is_open:
        last_cmd = "N"

    # العرض على الشاشة
    cv2.putText(
        frame,
        f"Preview: {command_name(preview_command)}",
        (20, 40),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.9,
        (0, 255, 0),
        2,
    )

    cv2.putText(
        frame,
        f"Will Send: {preview_command}",
        (20, 80),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.8,
        (255, 255, 0),
        2,
    )

    mouth_text = "OPEN" if mouth_is_open else "CLOSED"
    cv2.putText(
        frame,
        f"Mouth: {mouth_text}",
        (20, 120),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.8,
        (0, 200, 255),
        2,
    )

    if command_to_send != "N":
        cv2.putText(
            frame,
            f"Sent: {command_name(command_to_send)}",
            (20, 160),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.8,
            (0, 0, 255),
            2,
        )

    cv2.imshow("Gesture Control", frame)

    if cv2.waitKey(1) & 0xFF == 27:
        break

cap.release()
cv2.destroyAllWindows()
arduino.close()